In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

In [4]:
df = pd.read_csv(r"C:\Users\Lenovo\Downloads\employer_attrition_data.csv")

X = df.drop(columns=["employee_id", "attrition"])
y = df["attrition"]

num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object", "category"]).columns

print("Dataset shape:", df.shape)
print("Missing values:", df.isnull().sum().sum())
print("Attrition distribution:")
print(y.value_counts(normalize=True))

Dataset shape: (600, 12)
Missing values: 52
Attrition distribution:
attrition
0    0.833333
1    0.166667
Name: proportion, dtype: float64


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_23412\14341545.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=["object", "category"]).columns


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [6]:
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

In [7]:
models = {
    "Logistic Regression": LogisticRegression(
        class_weight="balanced",
        max_iter=1000
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=42
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        random_state=42
    ),

    "SVM": SVC(
        class_weight="balanced",
        probability=True,
        random_state=42
    )
}

In [14]:
results = []

for name, model in models.items():

    pipeline = Pipeline([
        ("preprocessing", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1 Score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="F1 Score",
    ascending=False
)

print(results_df.round(4))

                 Model  Accuracy  Precision  Recall  F1 Score  ROC-AUC
3                  SVM    0.7333     0.3421    0.65    0.4483   0.7535
0  Logistic Regression    0.7167     0.3250    0.65    0.4333   0.7485
1        Random Forest    0.8167     0.4167    0.25    0.3125   0.7090
2    Gradient Boosting    0.8417     0.5714    0.20    0.2963   0.7435


C:\Users\Lenovo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


In [15]:
best_model_name = results_df.iloc[0]["Model"]

print("Best model:", best_model_name)

Best model: SVM


In [16]:
best_model = Pipeline([
    ("preprocessing", preprocessor),
    ("model", models[best_model_name])
])

best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)

print(classification_report(
    y_test,
    y_pred,
    target_names=["Stayed", "Left"],
    zero_division=0
))

              precision    recall  f1-score   support

      Stayed       0.91      0.75      0.82       100
        Left       0.34      0.65      0.45        20

    accuracy                           0.73       120
   macro avg       0.63      0.70      0.64       120
weighted avg       0.82      0.73      0.76       120



C:\Users\Lenovo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


In [18]:
new_employee = pd.DataFrame([{
    "age": 25,
    "years_at_company": 2,
    "monthly_income_usd": 2500,
    "job_satisfaction": 2,
    "work_life_balance": 2,
    "distance_from_home_km": 15,
    "projects_completed": 4,
    "department": "IT",
    "education": "Bachelor",
    "overtime": "Yes"
}])
new_employee

,age,years_at_company,monthly_income_usd,job_satisfaction,work_life_balance,distance_from_home_km,projects_completed,department,education,overtime
0,25,2,2500,2,2,15,4,IT,Bachelor,Yes


In [19]:
prediction = best_model.predict(new_employee)

probability = best_model.predict_proba(new_employee)[0][1]

if prediction[0] == 1:
    print("Employee is likely to leave.")
else:
    print("Employee is likely to stay.")

print(f"Probability of leaving: {probability:.2%}")

Employee is likely to leave.
Probability of leaving: 45.08%
